# 01 - Fine-Tashkeel (ByT5) Inference

**Model:** `basharalrfooh/Fine-Tashkeel`  
**Architecture:** ByT5 (Byte-Level T5) - Seq2Seq  
**Reported Performance:** DER 0.95%, WER 2.49%

## Why Byte-Level?
- No vocabulary lookup - processes raw UTF-8 bytes
- Robust to rare words, dialects, spelling variations
- Preserves Arabic morphological patterns

**Tasks:**
1. Dev Set: Inference + Metrics (DER, WER, SER)
2. Blind Test: Inference + Submission JSON

In [1]:
!pip install -q transformers torch accelerate sentencepiece tqdm pyarabic

In [2]:
# Setup - Import from config.py (generated by setup.sh)
import os
import sys
import json
import re
import torch
import zipfile
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings
warnings.filterwarnings('ignore')

# Add project root to path for config import
IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))

# Import paths from config
try:
    from config import (
        PROJECT_DIR, DATA_DIR, DEVICE,
        DEV_AUDIO_DIR, DEV_TEXT_DIR,
        DEV_INPUT, DEV_OUTPUT,
        TEST_DIR, TEST_AUDIO_DIR,
        OUTPUT_DIR, SUBMISSION_DIR
    )
    TEST_INPUT = TEST_DIR / 'test_input.json'
except ImportError:
    print("WARNING: config.py not found. Run setup.sh first!")
    DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
    DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
    DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
    TEST_DIR = PROJECT_DIR / 'Test'
    TEST_INPUT = TEST_DIR / 'test_input.json'
    OUTPUT_DIR = PROJECT_DIR / 'outputs'
    SUBMISSION_DIR = PROJECT_DIR / 'submissions'
    DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

OUTPUT_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

device = DEVICE if 'DEVICE' in dir() else ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Environment: 'Local'")
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

Environment: Local
Device: cuda
GPU: NVIDIA RTX A5000
VRAM: 23.6 GB


In [3]:
# Model Configuration
MODEL_KEY = 'fine_tashkeel'
MODEL_NAME = 'basharalrfooh/Fine-Tashkeel'

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if device == "cuda":
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
    )
else:
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

model.eval()
print(f"Model loaded! Parameters: {sum(p.numel() for p in model.parameters()):,}")
if device == "cuda":
    print(f"GPU Memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Loading basharalrfooh/Fine-Tashkeel...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/308 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded! Parameters: 720,870,144
GPU Memory: 1.70 GB


In [4]:
@torch.inference_mode()
def diacritize(text, max_length=1024):
    """Diacritize Arabic text using Fine-Tashkeel."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length, padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_beams=4,
        early_stopping=True,
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

## Evaluation Metrics

In [5]:
DIACRITIC_PATTERN = re.compile(r'[\u064B-\u0652]')
ARABIC_LETTER_PATTERN = re.compile(r'[\u0621-\u064A]')
IRAB_DIACRITICS = {'\u064B', '\u064C', '\u064D'}

def extract_diacritics(text):
    result = []
    i = 0
    while i < len(text):
        char = text[i]
        if ARABIC_LETTER_PATTERN.match(char):
            diacritics = []
            j = i + 1
            while j < len(text) and DIACRITIC_PATTERN.match(text[j]):
                diacritics.append(text[j])
                j += 1
            result.append((char, ''.join(diacritics)))
            i = j
        else:
            i += 1
    return result

def compute_metrics(predictions, ground_truth, exclude_irab=False):
    gt_lookup = {item['id']: item['text_diacritized'] for item in ground_truth}
    total_chars, total_words, char_errors, word_errors, ser_sum = 0, 0, 0, 0, 0
    n = 0
    
    for pred in predictions:
        if pred['id'] not in gt_lookup:
            continue
        pred_text = pred['text_diacritized']
        ref_text = gt_lookup[pred['id']]
        
        pred_pairs = extract_diacritics(pred_text)
        ref_pairs = extract_diacritics(ref_text)
        
        # DER
        min_len = min(len(pred_pairs), len(ref_pairs))
        for i in range(min_len):
            pred_d, ref_d = pred_pairs[i][1], ref_pairs[i][1]
            if exclude_irab:
                pred_d = ''.join(d for d in pred_d if d not in IRAB_DIACRITICS)
                ref_d = ''.join(d for d in ref_d if d not in IRAB_DIACRITICS)
            if pred_d != ref_d:
                char_errors += 1
        char_errors += abs(len(pred_pairs) - len(ref_pairs))
        total_chars += max(len(pred_pairs), len(ref_pairs))
        
        # WER
        pred_words, ref_words = pred_text.split(), ref_text.split()
        for i in range(min(len(pred_words), len(ref_words))):
            if pred_words[i] != ref_words[i]:
                word_errors += 1
        word_errors += abs(len(pred_words) - len(ref_words))
        total_words += max(len(pred_words), len(ref_words))
        
        # SER
        if pred_text != ref_text:
            ser_sum += 1
        n += 1
    
    return {
        'DER': char_errors / total_chars if total_chars > 0 else 0,
        'WER': word_errors / total_words if total_words > 0 else 0,
        'SER': ser_sum / n if n > 0 else 0,
        'n_samples': n
    }

## Part 1: Dev Set Inference + Metrics

In [6]:
# Load dev data
with open(DEV_INPUT, 'r', encoding='utf-8') as f:
    dev_input = json.load(f)
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f:
    dev_output = json.load(f)

print(f"Dev samples: {len(dev_input)}")

Dev samples: 260


In [9]:
# Run inference on dev set
dev_predictions = []

for item in tqdm(dev_input, desc="Dev Set Inference"):
    try:
        diacritized = diacritize(item['text_undiacritized'])
        dev_predictions.append({'id': item['id'], 'text_diacritized': diacritized})
    except Exception as e:
        print(f"Error on {item['id']}: {e}")
        dev_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})

print(f"\nProcessed {len(dev_predictions)} samples")

Dev Set Inference: 100%|██████████| 260/260 [23:04<00:00,  5.32s/it]


Processed 260 samples


In [10]:
# Compute metrics
print("="*60)
print("DEV SET RESULTS")
print("="*60)

metrics_with_irab = compute_metrics(dev_predictions, dev_output, exclude_irab=False)
print(f"\n[Including I'rab (case endings)]")
print(f"  DER: {metrics_with_irab['DER']*100:.2f}%")
print(f"  WER: {metrics_with_irab['WER']*100:.2f}% (PRIMARY)")
print(f"  SER: {metrics_with_irab['SER']*100:.2f}%")

metrics_no_irab = compute_metrics(dev_predictions, dev_output, exclude_irab=True)
print(f"\n[Excluding I'rab (case endings)]")
print(f"  DER: {metrics_no_irab['DER']*100:.2f}%")
print(f"  WER: {metrics_no_irab['WER']*100:.2f}%")
print(f"  SER: {metrics_no_irab['SER']*100:.2f}%")

DEV SET RESULTS

[Including I'rab (case endings)]
  DER: 14.16%
  WER: 42.80% (PRIMARY)
  SER: 83.46%

[Excluding I'rab (case endings)]
  DER: 14.10%
  WER: 42.80%
  SER: 83.46%


In [11]:
# Save dev predictions
dev_pred_file = OUTPUT_DIR / f"{MODEL_KEY}_dev_predictions.json"
with open(dev_pred_file, 'w', encoding='utf-8') as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=2)
print(f"Dev predictions saved: {dev_pred_file}")

Dev predictions saved: ./outputs/fine_tashkeel_dev_predictions.json


## Part 2: Blind Test Inference + Submission

In [7]:
# Load test data
with open(TEST_INPUT, 'r', encoding='utf-8') as f:
    test_input = json.load(f)

print(f"Test samples: {len(test_input)}")

Test samples: 328


In [8]:
# Run inference on test set
test_predictions = []

for item in tqdm(test_input, desc="Test Set Inference"):
    try:
        diacritized = diacritize(item['text_undiacritized'])
        test_predictions.append({'id': item['id'], 'text_diacritized': diacritized})
    except Exception as e:
        print(f"Error on {item['id']}: {e}")
        test_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})

print(f"\nProcessed {len(test_predictions)} samples")

Test Set Inference: 100%|██████████| 328/328 [35:37<00:00,  6.52s/it]


Processed 328 samples


In [9]:
# Save test predictions
test_pred_file = OUTPUT_DIR / f"{MODEL_KEY}_test_predictions.json"
with open(test_pred_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)
print(f"Test predictions saved: {test_pred_file}")

Test predictions saved: ./outputs/fine_tashkeel_test_predictions.json


In [10]:
# Create submission ZIP
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

json_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

zip_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission_{timestamp}.zip"
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')

print("="*60)
print("SUBMISSION READY FOR CODABENCH")
print("="*60)
print(f"JSON: {json_file}")
print(f"ZIP:  {zip_file}")
print(f"Size: {zip_file.stat().st_size / 1024:.1f} KB")
print(f"Samples: {len(test_predictions)}")

SUBMISSION READY FOR CODABENCH
JSON: ./submissions/fine_tashkeel_submission.json
ZIP:  ./submissions/fine_tashkeel_submission_20260220_113612.zip
Size: 18.9 KB
Samples: 328


## Sample Comparisons

In [11]:
# Show sample comparisons from dev set
gt_lookup = {item['id']: item['text_diacritized'] for item in dev_output}

print("Sample Comparisons (Dev Set):")
print("="*80)
for i, pred in enumerate(dev_predictions[:5]):
    print(f"\n[{i+1}] ID: {pred['id']}")
    print(f"Pred:   {pred['text_diacritized'][:70]}...")
    print(f"Ground: {gt_lookup.get(pred['id'], 'N/A')[:70]}...")

Sample Comparisons (Dev Set):


NameError: name 'dev_predictions' is not defined

In [ ]:
# Google Drive sync removed for public release
